In [7]:
import numpy as np
import pandas as pd
import csv
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor as xgb
import math

In [8]:
dic={}
ref=0

#读取训练集数据并处理
def getdata(f):
    global ref,dic
    d=pd.read_csv(f)
    #print(d['Id'])
    d.drop(['Id'],axis=1,inplace=True)
    tmphead=list(d.head())
    tmplist=list(d.values)
    rlist=[]
    #对训练集中的字符串数据进行处理
    for i in tmplist:
        tmp=[]
        for t in i:
            try:
                if(math.isnan(float(t))):
                    #用字典dic把字符串类型的数据映射到float上
                    try:
                        tmp.append(dic['NA'])
                    except:
                        dic['NA']=ref
                        ref+=1
                        tmp.append(dic['NA'])
                else:
                    tmp.append(float(t))
            except:
                try:
                    tmp.append(dic[t])
                except:
                    dic[t]=ref
                    ref+=1
                    tmp.append(dic[t])
        rlist.append(tmp)
    return pd.DataFrame(rlist, columns=tmphead)

#读取测试集数据并处理
def gettarget(f):
    global dic
    d=pd.read_csv(f)
    tmphead=list(d.head())
    tmplist=list(d.values)
    rlist=[] 
    for i in tmplist:
        tmp=[]
        for t in i:
            try:
                if(math.isnan(float(t))):
                    tmp.append(dic['NA'])
                else:
                    tmp.append(float(t))
            except:
                tmp.append(dic[t])
        rlist.append(tmp)
        
    return pd.DataFrame(rlist, columns=tmphead)


In [9]:
#文件写入操作
def writedata(idi,data,file):
    with open(file,'w',newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['Id','SalePrice'])
        for i in range(len(data)):
            writer.writerow([int(idi[i]),data[i]])
            
def getresult(mymodel,target):
    return mymodel.predict(target)

#选择模型最优参数并训练、预测    


In [10]:
d=getdata('train.csv')      #getdata返回经处理过的数据
corrmat = d.corr()          #计算相关系数
rela=list(corrmat['SalePrice'].abs().sort_values().index)[:-1]      #将特征列按相关性大小排序
features=68
select_feat=rela[-features:]            #取相关性好的前68列

In [11]:
train,label=d.drop(['SalePrice'],axis=1,inplace=False),d['SalePrice']
train=train[select_feat]
d=gettarget('test.csv')
idi,target=d['Id'],d.drop(['Id'],axis=1,inplace=False)
target=target[select_feat]

In [13]:
def selectmodel(train, label):
    # 数据清理
    mask = ~np.isnan(train).any(axis=1) & ~np.isnan(label)
    train_clean = train[mask]
    label_clean = label[mask]

    # 检查清理后的数据
    if np.isnan(train_clean).any() or np.isnan(label_clean).any():
        raise ValueError("数据清理后仍存在NaN值")

    # 原有的模型训练代码
    params = {'booster': ['gbtree', 'gblinear', 'dart']}
    mymodel = GridSearchCV(xgb(), params, error_score=np.nan, refit=True)
    mymodel.fit(train_clean, label_clean)
    return mymodel, mymodel.best_score_

In [19]:
try:
	# Train the model and get the best model
	mymodel, best_score = selectmodel(train, label)
	print(f"Model trained successfully with best score: {best_score}")

	# Get predictions
	result = getresult(mymodel, target)
	print("Predictions generated successfully.")

	# Write predictions to CSV
	writedata(idi, result, 'submission1.csv')
	print("Results written to 'submission1.csv' successfully.")
except Exception as e:
	print(f"An error occurred: {e}")

An error occurred: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
